In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

files = [
    "Cherkasy_commercial.csv", "Chernihiv_commercial.csv", "Chernivtsi_commercial.csv", "Dnipro_commercial.csv", "Ivano-Frankivsk_commercial.csv",
    "Kharkiv_commercial.csv", "Khmelnytskyi_commercial.csv", "Kropyvnytskyi_commercial.csv",
    "Lutsk_commercial.csv", "Lviv_commercial.csv", "Mykolaiv_commercial.csv", "Odesa_commercial.csv", "Poltava_commercial.csv", "Rivne_commercial.csv",
    "Sumy_commercial.csv", "Ternopil_commercial.csv", "Uzhhorod_commercial.csv", "Vinnytsia_commercial.csv", "Zaporizhzhia_commercial.csv",
    "Zhytomyr_commercial.csv"
]

dfs = []
for filename in files:
    df = pd.read_csv(filename)
    dfs.append(df)
    print(f"✓ {filename}")

df_ukraine = pd.concat(dfs, ignore_index=True)

df_kyiv = pd.read_csv("kyiv_commercial.csv")

✓ Cherkasy_commercial.csv
✓ Chernihiv_commercial.csv
✓ Chernivtsi_commercial.csv
✓ Dnipro_commercial.csv
✓ Ivano-Frankivsk_commercial.csv
✓ Kharkiv_commercial.csv
✓ Khmelnytskyi_commercial.csv
✓ Kropyvnytskyi_commercial.csv
✓ Lutsk_commercial.csv
✓ Lviv_commercial.csv
✓ Mykolaiv_commercial.csv
✓ Odesa_commercial.csv
✓ Poltava_commercial.csv
✓ Rivne_commercial.csv
✓ Sumy_commercial.csv
✓ Ternopil_commercial.csv
✓ Uzhhorod_commercial.csv
✓ Vinnytsia_commercial.csv
✓ Zaporizhzhia_commercial.csv
✓ Zhytomyr_commercial.csv


In [2]:
df_ukraine.isna().sum()

group_id                     0
has_duplicates               0
url                          0
city                         0
district                  1679
residential_complex       1088
lat                          0
lon                          0
distance_to_center_km        0
poi_name                   349
poi_distance_m             349
geo_region                   0
price                        0
area                         0
floor                     1670
floor_count                598
house_type                 749
wall_type                  827
heating                    786
ceiling_height             847
year_of_building           660
is_bank                      0
is_office                    0
is_services                  0
is_warehouse                 0
is_production                0
is_free                      0
is_retail                    0
is_garage                    0
is_parking_spot              0
has_tenant                   0
rental_price              1670
implied_

In [3]:
df_kyiv.isna().sum()

group_id                     0
has_duplicates               0
url                          0
city                         0
district                     0
residential_complex       1549
lat                          0
lon                          0
distance_to_center_km        0
poi_name                  1952
poi_distance_m            1952
geo_region                   0
price                        0
area                         0
floor                     2817
floor_count                262
house_type                 265
wall_type                  261
heating                    256
ceiling_height             262
year_of_building           231
is_bank                      0
is_office                    0
is_services                  0
is_warehouse                 0
is_production                0
is_free                      0
is_retail                    0
is_garage                    0
is_parking_spot              0
has_tenant                   0
rental_price              2773
implied_

In [4]:
PRICE_MIN, PRICE_MAX = 50, 20_000_000


def clean_price_and_area(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df[df["price"].notna() & df["price"].between(PRICE_MIN, PRICE_MAX)]
    df = df[df["area"].notna() & (df["area"] > 0)]
    df = df[df["lat"].notna() & df["lon"].notna() & df["distance_to_center_km"].notna()]
    print(f"{name}: прибрано {before - len(df)} рядків (ціна/площа/координати). Лишилось: {len(df)}")
    return df


df_ukraine = clean_price_and_area(df_ukraine, "Ukraine")
df_kyiv = clean_price_and_area(df_kyiv, "Kyiv")

Ukraine: прибрано 1 рядків (ціна/площа/координати). Лишилось: 1678
Kyiv: прибрано 5 рядків (ціна/площа/координати). Лишилось: 2820


In [5]:
PRICE_SQM_MIN, PRICE_SQM_MAX = 5, 20000


def apply_price_sqm_sanity_filter(df: pd.DataFrame, name: str, low: float = PRICE_SQM_MIN, high: float = PRICE_SQM_MAX) -> pd.DataFrame:
    before = len(df)
    ratio = df["price"] / df["area"]
    df = df[ratio.between(low, high)]
    print(f"{name}: прибрано {before - len(df)} рядків (неправдоподібний $/м²). Лишилось: {len(df)}")
    return df


df_ukraine = apply_price_sqm_sanity_filter(df_ukraine, "Ukraine")
df_kyiv = apply_price_sqm_sanity_filter(df_kyiv, "Kyiv")

Ukraine: прибрано 1 рядків (неправдоподібний $/м²). Лишилось: 1677
Kyiv: прибрано 12 рядків (неправдоподібний $/м²). Лишилось: 2808


In [6]:
def object_group(df: pd.DataFrame) -> pd.Series:
    is_parking = df["is_garage"].astype(bool) | df["is_parking_spot"].astype(bool)
    is_warehouse = df["is_warehouse"].astype(bool)
    return pd.Series(
        np.select([is_parking, is_warehouse], ["parking_or_garage", "warehouse"], default="premises"),
        index=df.index,
    )


def remove_outliers_iqr_grouped(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    groups = object_group(df)
    keep_mask = pd.Series(True, index=df.index)

    for group_value in pd.unique(groups):
        idx = df.index[groups == group_value]
        group_df = df.loc[idx]
        group_mask = pd.Series(True, index=idx)
        for col in cols:
            q1, q3 = group_df[col].quantile(0.25), group_df[col].quantile(0.75)
            iqr = q3 - q1
            lower, upper = q1 - k * iqr, q3 + k * iqr
            group_mask &= group_df[col].between(lower, upper)
            print(f"  [{group_value}] {col}: lower={lower:.1f}, upper={upper:.1f}")
        keep_mask.loc[idx] = group_mask

    return df[keep_mask]

In [7]:
before = len(df_ukraine)
df_ukraine = remove_outliers_iqr_grouped(df_ukraine, ["price", "area"])
print(f"\nПрибрано {before - len(df_ukraine)} викидів по Україні. Лишилось: {len(df_ukraine)}")

  [parking_or_garage] price: lower=-4500.0, upper=35500.0
  [parking_or_garage] area: lower=4.5, upper=34.1
  [warehouse] price: lower=-305225.0, upper=655375.0
  [warehouse] area: lower=-1042.0, upper=1936.6
  [premises] price: lower=-184000.0, upper=464000.0
  [premises] area: lower=-100.0, upper=300.0

Прибрано 227 викидів по Україні. Лишилось: 1450


In [8]:
before = len(df_kyiv)
df_kyiv = remove_outliers_iqr_grouped(df_kyiv, ["price", "area"])
print(f"\nПрибрано {before - len(df_kyiv)} викидів в Києві. Лишилось: {len(df_kyiv)}")

  [premises] price: lower=-410000.0, upper=1030000.0
  [premises] area: lower=-144.7, upper=427.6
  [warehouse] price: lower=-877500.0, upper=1726500.0
  [warehouse] area: lower=-662.5, upper=1309.5
  [parking_or_garage] price: lower=-7500.0, upper=76500.0
  [parking_or_garage] area: lower=-0.7, upper=38.5

Прибрано 394 викидів в Києві. Лишилось: 2414


In [9]:
for df in (df_ukraine, df_kyiv):
    df["in_residential_complex"] = df["residential_complex"].notna().astype(int)
    df["has_poi"] = df["poi_name"].notna().astype(int)
    df.drop(columns=["residential_complex", "poi_name"], inplace=True)

In [10]:
for df in (df_ukraine, df_kyiv):
    df.drop(columns=["floor"], inplace=True)

In [11]:
NUMERIC_TARGETS = ["year_of_building", "ceiling_height", "floor_count"]
PURPOSE_COLS = [
    "is_bank", "is_office", "is_services", "is_warehouse",
    "is_production", "is_free", "is_retail", "is_garage", "is_parking_spot",
]


def impute_numeric_grouped(df: pd.DataFrame, location_cols: list[str]) -> pd.DataFrame:
    groups = object_group(df)
    for group_value in pd.unique(groups):
        idx = df.index[groups == group_value]
        subset = df.loc[idx]

        location_onehot = pd.get_dummies(subset[location_cols], columns=location_cols)
        numeric_cols = ["area"] + NUMERIC_TARGETS
        numeric_source = subset[numeric_cols].copy()

        scaler = StandardScaler()
        scaled_numeric = pd.DataFrame(
            scaler.fit_transform(numeric_source), columns=numeric_cols, index=idx,
        )
        knn_input = pd.concat(
            [scaled_numeric,
             subset[PURPOSE_COLS].reset_index(drop=True).set_axis(idx),
             location_onehot.reset_index(drop=True).set_axis(idx)],
            axis=1,
        )
        imputer = KNNImputer(n_neighbors=5, weights="distance")
        imputed_array = imputer.fit_transform(knn_input)
        imputed = pd.DataFrame(imputed_array, columns=knn_input.columns, index=idx)
        imputed_numeric = pd.DataFrame(
            scaler.inverse_transform(imputed[numeric_cols]), columns=numeric_cols, index=idx,
        )

        df.loc[idx, "year_of_building"] = imputed_numeric["year_of_building"].round().astype("Int64")
        df.loc[idx, "ceiling_height"] = imputed_numeric["ceiling_height"].round(1)
        df.loc[idx, "floor_count"] = imputed_numeric["floor_count"].round().astype("Int64")

        print(f"  [{group_value}] імпутовано {len(idx)} рядків")
    return df

In [12]:
df_ukraine = impute_numeric_grouped(df_ukraine, location_cols=["district", "city"])
df_ukraine[NUMERIC_TARGETS].isna().sum()

  [parking_or_garage] імпутовано 263 рядків
  [warehouse] імпутовано 698 рядків
  [premises] імпутовано 489 рядків


year_of_building    0
ceiling_height      0
floor_count         0
dtype: int64

In [13]:
df_kyiv = impute_numeric_grouped(df_kyiv, location_cols=["district"])
df_kyiv[NUMERIC_TARGETS].isna().sum()

  [premises] імпутовано 1687 рядків
  [warehouse] імпутовано 395 рядків
  [parking_or_garage] імпутовано 332 рядків


year_of_building    0
ceiling_height      0
floor_count         0
dtype: int64

In [14]:
for df in (df_ukraine, df_kyiv):
    df["poi_distance_m"] = df["poi_distance_m"].fillna(df["poi_distance_m"].median())

In [15]:
for df in (df_ukraine, df_kyiv):
    for col in ["house_type", "wall_type", "heating"]:
        df[col] = df[col].fillna("Unknown")

df_ukraine[["house_type", "wall_type", "heating"]].isna().sum()

house_type    0
wall_type     0
heating       0
dtype: int64

In [16]:
df_kyiv[["house_type", "wall_type", "heating"]].isna().sum()

house_type    0
wall_type     0
heating       0
dtype: int64

In [17]:
df_ukraine["price"] = df_ukraine["price"].round(0)
df_ukraine["area"] = df_ukraine["area"].round(1)
df_ukraine[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,1450.000000,1450.000000,1450.000000
mean,113682.311034,188.583448,2.799793
std,119690.223883,314.728047,0.213243
min,120.000000,1.000000,2.400000
25%,26000.000000,30.000000,2.700000
50%,72500.000000,78.100000,2.800000
75%,160000.000000,164.625000,2.800000
max,654375.000000,1900.000000,4.000000


In [18]:
df_kyiv["price"] = df_kyiv["price"].round(0)
df_kyiv["area"] = df_kyiv["area"].round(1)
df_kyiv[["price", "area", "ceiling_height"]].describe()

,price,area,ceiling_height
count,2.414000e+03,2414.000000,2414.000000
mean,2.386303e+05,121.629246,2.913380
std,2.297353e+05,122.751529,0.348698
min,2.000000e+03,2.200000,2.000000
25%,7.660000e+04,46.000000,2.700000
50%,1.650000e+05,88.000000,2.800000
75%,3.300000e+05,156.000000,3.000000
max,1.670000e+06,1296.000000,5.000000


In [19]:
df_kyiv.to_csv("Kyiv_commercial_for_analysis.csv", index=False)
df_ukraine.to_csv("Ukraine_commercial_for_analysis.csv", index=False)

In [20]:
before = len(df_ukraine)
df_encoded_ukraine = df_ukraine.drop_duplicates(subset="group_id", keep="first").copy()
print(f"Прибрано {before - len(df_encoded_ukraine)} дублікатів за group_id (Україна)")

df_encoded_ukraine = df_encoded_ukraine.drop(columns=["url", "group_id", "geo_region", "text", "has_duplicates", "lat", "lon"])

categorical_cols_ukraine = ["house_type", "wall_type", "heating", "city"]
df_encoded_ukraine = df_encoded_ukraine.drop(columns=["district"])
df_encoded_ukraine = pd.get_dummies(df_encoded_ukraine, columns=categorical_cols_ukraine, drop_first=True)
df_encoded_ukraine.to_csv("Ukraine_commercial_ML.csv", index=False)
df_encoded_ukraine.shape

Прибрано 6 дублікатів за group_id (Україна)


(1444, 69)

In [21]:
before = len(df_kyiv)
df_encoded_kyiv = df_kyiv.drop_duplicates(subset="group_id", keep="first").copy()
print(f"Прибрано {before - len(df_encoded_kyiv)} дублікатів за group_id (Київ)")

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["url", "group_id", "geo_region", "text", "has_duplicates", "lat", "lon"])

df_encoded_kyiv = df_encoded_kyiv.drop(columns=["city"])
categorical_cols_kyiv = ["house_type", "wall_type", "heating", "district"]
df_encoded_kyiv = pd.get_dummies(df_encoded_kyiv, columns=categorical_cols_kyiv, drop_first=True)
df_encoded_kyiv.to_csv("Kyiv_commercial_ML.csv", index=False)
df_encoded_kyiv.shape

Прибрано 115 дублікатів за group_id (Київ)


(2299, 65)